## Feature check: six columns the EDA track hasn't tested

Six features are in our model but not on the EDA track's committed list. Four of
them were never analyzed on either side, so this settles them with one run.

**What it does**

- Prevalence and raw default lift for each, to catch anything too small to matter.
- Adds all six to the EDA track's adjusted regression, so each one's effect is
  measured with FICO, LTV, DTI, and everything else held fixed.

**Why the adjusted number is the one that counts.** Mortgage insurance is required
above 80 LTV, so its raw default rate mostly measures LTV. The EDA track's own
regression already shows this: `has_mi` looks meaningful raw and lands at OR 1.11
adjusted, which is close to nothing.

**Decision rule, set before looking:** odds ratio outside 0.87 to 1.15 stays.
Anything inside gets cut, no follow-up.

In [1]:
import polars as pl
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression

PROC = Path("../data/processed")
df = pl.read_parquet(PROC / "fannie_2017_typed.parquet")

BASE = df["default_flag"].mean()
print(f"rows: {len(df):,}   base default rate: {BASE:.4f}\n")

# ---- build the six candidates ----
df = df.with_columns([
    (pl.col("OCLTV") - pl.col("OLTV")).alias("second_lien"),
    pl.col("MI_TYPE").fill_null("NONE"),
    pl.col("CSCORE_C").is_not_null().cast(pl.Int8).alias("has_coborrower"),
    (pl.col("MI_TYPE").is_not_null()).cast(pl.Int8).alias("has_mi"),
    (pl.col("HIGH_BALANCE_LOAN_INDICATOR") == "Y").cast(pl.Int8).alias("is_high_balance"),
    (pl.col("PROPERTY_INSPECTION_WAIVER_INDICATOR") == "Y").cast(pl.Int8).alias("is_piw"),
    (pl.col("RELOCATION_MORTGAGE_INDICATOR") == "Y").cast(pl.Int8).alias("is_relo"),
])

# ---- 1. prevalence and raw lift ----
print("--- prevalence and raw lift ---")
print(f"{'feature':22} {'n flagged':>12} {'share':>8} {'def rate':>9} {'lift':>7}")

FLAGS = ["has_mi", "is_high_balance", "is_piw", "is_relo"]
for c in FLAGS:
    sub = df.filter(pl.col(c) == 1)
    n = len(sub)
    r = sub["default_flag"].mean() if n else float("nan")
    print(f"{c:22} {n:>12,} {n/len(df):>7.2%} {r:>9.4f} {r/BASE:>7.2f}")

sub = df.filter(pl.col("second_lien") > 0)
r = sub["default_flag"].mean()
print(f"{'second_lien > 0':22} {len(sub):>12,} {len(sub)/len(df):>7.2%} {r:>9.4f} {r/BASE:>7.2f}")

print("\n--- MI_TYPE levels ---")
print(
    df.group_by("MI_TYPE")
      .agg(pl.len().alias("n"), pl.col("default_flag").mean().round(4).alias("def_rate"))
      .with_columns((pl.col("def_rate") / BASE).round(2).alias("lift"))
      .sort("n", descending=True)
)

# ---- 2. adjusted regression: EDA track's model + the six ----
NUM = ["CSCORE_B", "DTI", "OLTV", "ORIG_UPB", "ORIG_TERM", "NUM_BO", "second_lien"]
FLG = ["has_coborrower", "is_hfa", "is_first_time", "is_homeready",
       "is_high_balance", "is_piw", "is_relo"]

model_df = (
    df.select(NUM + FLG + ["MI_TYPE", "default_flag"])
      .drop_nulls(subset=["CSCORE_B", "DTI"])
      .to_pandas()
)
print(f"\nregression rows (complete cases): {len(model_df):,}")

X = model_df[NUM + FLG].copy()
X[NUM] = (X[NUM] - X[NUM].mean()) / X[NUM].std()          # standardize numerics
mi = pd.get_dummies(model_df["MI_TYPE"], prefix="MI").drop(columns=["MI_NONE"])
X = pd.concat([X, mi.astype(float)], axis=1)
y = model_df["default_flag"].values

fit = LogisticRegression(max_iter=1000, solver="lbfgs", C=1e6).fit(X, y)

out = (
    pd.DataFrame({"feature": X.columns, "coef": fit.coef_[0]})
      .assign(odds_ratio=lambda d: np.exp(d["coef"]).round(3))
      .sort_values("odds_ratio")
      .reset_index(drop=True)
)

CANDIDATES = ["second_lien", "is_high_balance", "is_piw", "is_relo"] + list(mi.columns)
out["candidate"] = out["feature"].isin(CANDIDATES)
out["verdict"] = np.where(
    ~out["candidate"], "",
    np.where((out["odds_ratio"] < 0.87) | (out["odds_ratio"] > 1.15), "KEEP", "cut"),
)

print("\n--- adjusted odds ratios (all others held fixed) ---")
print(f"{'feature':22} {'OR':>8}  {'':4} {'verdict':>8}")
for _, r in out.iterrows():
    mark = "<--" if r["candidate"] else "   "
    print(f"{r['feature']:22} {r['odds_ratio']:>8.3f}  {mark} {r['verdict']:>8}")

print("\nRule: OR outside 0.87-1.15 stays. Inside gets cut.")
print("MI_ levels are vs. no mortgage insurance. If they cluster together,")
print("'who pays' carries nothing and MI_TYPE reduces to has_mi.")

rows: 2,046,851   base default rate: 0.0341

--- prevalence and raw lift ---
feature                   n flagged    share  def rate    lift
has_mi                      614,740  30.03%    0.0475    1.39
is_high_balance              53,388   2.61%    0.0601    1.76
is_piw                            0   0.00%       nan     nan
is_relo                      13,166   0.64%    0.0122    0.36
second_lien > 0              86,974   4.25%    0.0649    1.90

--- MI_TYPE levels ---
shape: (3, 4)
┌─────────┬─────────┬──────────┬──────┐
│ MI_TYPE ┆ n       ┆ def_rate ┆ lift │
│ ---     ┆ ---     ┆ ---      ┆ ---  │
│ str     ┆ u32     ┆ f64      ┆ f64  │
╞═════════╪═════════╪══════════╪══════╡
│ NONE    ┆ 1432111 ┆ 0.0284   ┆ 0.83 │
│ 1       ┆ 542264  ┆ 0.0483   ┆ 1.41 │
│ 2       ┆ 72476   ┆ 0.0412   ┆ 1.21 │
└─────────┴─────────┴──────────┴──────┘

regression rows (complete cases): 2,044,946

--- adjusted odds ratios (all others held fixed) ---
feature                      OR        verdict
has_co

## Three open feature questions

Follow-up to the six-candidate check. All three run on the same regression.

- **fico_missing**: is the flag worth a slot? 1,573 loans have no score. We either
  drop them or fill the score with the median and flag it. If the flag's odds
  ratio sits near 1.0, it carries nothing.
- **NO_UNITS**: in our model but never tested by either track. The only feature on
  the list with no evidence behind it.
- **is_high_balance**: came back at OR 2.014, the strongest thing outside FICO.
  High-balance loans concentrate in expensive coastal markets, so some of that
  could be geography. Refit with STATE controlled and see what survives.

In [2]:
import polars as pl
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

PROC = Path("../data/processed")
raw = pl.read_parquet(PROC / "fannie_2017_typed.parquet")

NUM = ["CSCORE_B", "DTI", "OLTV", "ORIG_UPB", "ORIG_TERM", "NUM_BO"]
FLG = ["is_first_time", "is_homeready", "is_hfa"]

def prep(df, add_extra=None, with_state=False):
    """Standardize numerics, add flags, optionally one-hot STATE."""
    cols = NUM + FLG + (add_extra or [])
    X = df.select(cols).to_pandas()
    X[NUM] = (X[NUM] - X[NUM].mean()) / X[NUM].std()
    if with_state:
        st = pd.get_dummies(df["STATE"].to_pandas(), prefix="ST", drop_first=True)
        X = pd.concat([X.reset_index(drop=True), st.astype(float).reset_index(drop=True)], axis=1)
    return X, df["default_flag"].to_numpy()

def fit_report(X, y, label, show=None):
    m = LogisticRegression(max_iter=2000, solver="lbfgs", C=1e6).fit(X, y)
    auc = roc_auc_score(y, m.predict_proba(X)[:, 1])
    print(f"  {label:34} n={len(y):>10,}   AUC {auc:.5f}")
    if show:
        ors = dict(zip(X.columns, np.exp(m.coef_[0])))
        for f in show:
            if f in ors:
                print(f"      {f:26} OR {ors[f]:.3f}")
    return auc

base = raw.with_columns([
    pl.col("CSCORE_C").is_not_null().cast(pl.Int8).alias("has_coborrower"),
    pl.col("CSCORE_B").is_null().cast(pl.Int8).alias("fico_missing"),
    (pl.col("HIGH_BALANCE_LOAN_INDICATOR") == "Y").cast(pl.Int8).alias("is_high_balance"),
    (pl.col("RELOCATION_MORTGAGE_INDICATOR") == "Y").cast(pl.Int8).alias("is_relo"),
    pl.col("NO_UNITS").cast(pl.Int32, strict=False),
]).drop_nulls(subset=["DTI"])

EXTRA = ["has_coborrower", "is_high_balance", "is_relo"]

# ============ 1. fico_missing ============
print("=== 1. fico_missing: drop vs. impute + flag ===")

dropped = base.drop_nulls(subset=["CSCORE_B"])
X, y = prep(dropped, EXTRA)
fit_report(X, y, "drop the no-score loans")

med = base["CSCORE_B"].median()
imputed = base.with_columns(pl.col("CSCORE_B").fill_null(med))
X, y = prep(imputed, EXTRA + ["fico_missing"])
fit_report(X, y, "median impute + fico_missing flag", show=["fico_missing"])

print("\n  If the flag's OR is near 1.0 it carries nothing. AUCs will be")
print("  nearly identical either way: this is 0.08% of the book.")

# ============ 2. NO_UNITS ============
print("\n=== 2. NO_UNITS ===")
BASE_RATE = base["default_flag"].mean()
print(
    base.group_by("NO_UNITS")
        .agg(pl.len().alias("n"), pl.col("default_flag").mean().round(4).alias("def_rate"))
        .with_columns((pl.col("def_rate") / BASE_RATE).round(2).alias("lift"))
        .sort("NO_UNITS")
)

X, y = prep(imputed, EXTRA + ["fico_missing", "NO_UNITS"])
X["NO_UNITS"] = (X["NO_UNITS"] - X["NO_UNITS"].mean()) / X["NO_UNITS"].std()
fit_report(X, y, "with NO_UNITS", show=["NO_UNITS"])

# ============ 3. is_high_balance, geography controlled ============
print("\n=== 3. is_high_balance with STATE controlled ===")

X, y = prep(imputed, EXTRA + ["fico_missing"], with_state=False)
fit_report(X, y, "without STATE", show=["is_high_balance"])

X, y = prep(imputed, EXTRA + ["fico_missing"], with_state=True)
fit_report(X, y, "with STATE", show=["is_high_balance"])

print("\n  If the OR holds near 2.0, high balance is real. If it drops toward 1.2,")
print("  it was mostly telling us the loan sits in an expensive market.")
print("\nRule: OR outside 0.87-1.15 stays.")

=== 1. fico_missing: drop vs. impute + flag ===
  drop the no-score loans            n= 2,044,946   AUC 0.75338
  median impute + fico_missing flag  n= 2,046,511   AUC 0.75331
      fico_missing               OR 1.018

  If the flag's OR is near 1.0 it carries nothing. AUCs will be
  nearly identical either way: this is 0.08% of the book.

=== 2. NO_UNITS ===
shape: (4, 4)
┌──────────┬─────────┬──────────┬──────┐
│ NO_UNITS ┆ n       ┆ def_rate ┆ lift │
│ ---      ┆ ---     ┆ ---      ┆ ---  │
│ i32      ┆ u32     ┆ f64      ┆ f64  │
╞══════════╪═════════╪══════════╪══════╡
│ 1        ┆ 2004015 ┆ 0.0338   ┆ 0.99 │
│ 2        ┆ 29744   ┆ 0.0499   ┆ 1.46 │
│ 3        ┆ 6435    ┆ 0.0606   ┆ 1.78 │
│ 4        ┆ 6317    ┆ 0.05     ┆ 1.46 │
└──────────┴─────────┴──────────┴──────┘
  with NO_UNITS                      n= 2,046,511   AUC 0.75445
      NO_UNITS                   OR 1.080

=== 3. is_high_balance with STATE controlled ===
  without STATE                      n= 2,046,511   AUC 0.

## Finding: three more cuts, one feature confirmed

Follow-up to the six-candidate check. Same regression, same rule: adjusted odds
ratio outside 0.87 to 1.15 stays, inside gets cut.

| Question | Result | Verdict |
|---|---|---|
| fico_missing flag | OR 1.018 | Cut |
| NO_UNITS | OR 1.080 | Cut |
| is_high_balance, geography controlled | OR 2.028 → 1.930 | Keep |

### fico_missing: cut the flag, keep the loans

1,573 loans have no credit score. We fill the score with the median and add a flag
so the model knows the value is fabricated.

The flag does nothing. OR 1.018, and dropping the loans entirely versus imputing
plus flagging gives AUCs seven hundred-thousandths apart.

Keep the median imputation so the loans stay in the book. Drop the flag.

We also considered filling with a sentinel value like 999 or 0 instead. That does
not work: the model reads any number you put there as a credit score, so a high
sentinel makes those loans look like top credit and a low one makes them look like
the worst. There is no number that means "not a number." The flag was the correct
version of that idea, it just turned out not to be worth a slot.

### NO_UNITS: cut

The only feature on our list with no evidence behind it from either track.

Raw lift looked real: single-unit properties default at the base rate, while 2-,
3-, and 4-unit properties run 1.46 to 1.78 times higher. But adjusted it lands at
1.080.

The reason is that multi-unit properties are usually investment purchases, and
`OCC_STAT` already carries occupancy. Once that and LTV are in the model, unit
count adds nothing.

### is_high_balance: confirmed

This came back at OR 2.014 in the first check, strong enough to be worth
questioning. High-balance loans concentrate in expensive coastal markets, so the
concern was that it might be a geography proxy rather than a property of the loan.

Refit with all 53 states as controls: 2.028 drops to 1.930. A 5% reduction. It is
not geography.

So high balance nearly doubles the odds of default with credit score, LTV, DTI,
loan size, and location all held fixed. It is the strongest non-credit feature in
the model and neither track had tested it.

**Side observation.** Adding STATE lifted overall AUC from 0.7533 to 0.7632, the
largest single jump across any of these tests. Geography carries real independent
signal, which matches the EDA track's Cramér's V ranking where State outranked LTV.

---

## Suggested final feature list: 18

**Numeric (6)**
CSCORE_B, CSCORE_C, OLTV, DTI, ORIG_UPB, ORIG_TERM

**Count (1)**
NUM_BO

**Flags (6)**
is_first_time, is_homeready, is_hfa, has_coborrower, is_high_balance, is_relo

**Categorical (5)**
CHANNEL, PURPOSE, PROP, OCC_STAT, STATE

### How we got here

Started at 26. Cut eight, added one.

**Cut for leakage or identity (4):** ORIG_RATE, priced off the same risk we are
predicting. SELLER, lender identity rather than borrower risk.

**Cut for redundancy (4):** OCLTV at 0.98 correlation with OLTV. MI_PCT, nearly a
restatement of LTV above 80. MI_TYPE and has_mi, since the paid-by levels landed
nine thousandths apart, meaning who pays carries nothing. NO_UNITS, absorbed by
OCC_STAT.

**Cut for no signal (2):** second_lien, our own addition, OR 1.074. fico_missing,
OR 1.018.

**Cut for empty column (1):** PROPERTY_INSPECTION_WAIVER, zero rows populated in
the 2017 book.

**Added (1):** is_high_balance at OR 1.930.

### The pattern worth naming

Raw default lift would have kept three features that the adjusted view rejected:
second_lien at 1.90 raw, high balance at 1.76, mortgage insurance at 1.39. Only one
of those three survived.

It also would have missed nothing, but it would have cost us three feature slots on
columns that carry no independent signal. Running new features through the full
model rather than a lift table is what separated them.